# Demo: MUSHRA-like 2D Experiment

To get started, download this repository navigate to the folder in which you downloaded it and
```bash
pip install -e .
```
If you did this, you can run the example below.

In [1]:
import whispy
from pathlib import Path

## Define the experiment through configuration files

Config files are used to define an experiment. For modularity, multiple configs are used. They can be changed, to change the experiment.

In [2]:
# directory containing the config files
config_dir = Path('..') / 'configs'

# This defines the attributes/qualities for the experiment.
config_attributes = config_dir / 'attributes.yml'

# This defines the stimuli used in the experiment
config_stimuli = config_dir / 'stimuli.yml'

# This defines the course of the experiment. The content here works hand in
# hand with the attributes and stimuli configs
config_experiment = config_dir / 'experiment.yml'

# This is the configuration of the rating GUI that is used in this example
config_mushra_like_2d = config_dir / 'mushra_like_2d.yml'


## Load the stimuli handler

Different 'Handlers' can be used to control stimuli playback. This example uses audio files stored in wav-files. This can be handled by the `SounddeviceHandler`

In [3]:
# directory containing the audio files
stimuli_dir = Path('..') / 'stimuli'

# create the handler
stimuli_handler = whispy.interfaces.SounddeviceHandler(
    config_stimuli, stimuli_dir, loop=True)

each handler has a `play` and `stop` method. It gets the stimulus name, which is defined in `config_stimuli`. In this case the stimulus names are `1`, `2` and `3`. Comment out an run the below to play around with this.

In [4]:
# play stimulus 1
#stimuli_handler.play(1)

# stop stimulus 1
# stimuli_handler.stop(1)

## Create the experimental course

An experiment often consists of multiple conditions and in addition multiple test might be performed or multiple attributes rated. This is defined in `config_experiment`.

In most cases, parts of the experiment are randomized and each participant of the experiments performs the tasks in a different order. This is done with the `experiments.course` function.

In [5]:
# All parts of the experiment are randomized using the default parameter values
experimental_course = whispy.experiments.course(config_experiment)

The print below shows the result. Each row of `experimental_course` is a task in the experiment. The blocks and sections are defined in `config_experiments`.

In [6]:
_ = [print(task) for task in experimental_course]

{'block': 0, 'section': 0, 'reference': 1, 'test': array([2, 3]), 'block_changed': True, 'section_changed': True, 'attribute': 'difference', 'block_name': 'Block 1', 'section_name': 'Section 1'}
{'block': 0, 'section': 1, 'reference': 1, 'test': array([3, 2]), 'block_changed': False, 'section_changed': True, 'attribute': 'difference', 'block_name': 'Block 1', 'section_name': 'Section 2'}
{'block': 1, 'section': 0, 'reference': 1, 'test': array([3, 2]), 'block_changed': True, 'section_changed': False, 'attribute': 'tone_color_bright-dark', 'block_name': 'Block 2', 'section_name': 'Section 3'}


## Run the experiment

A little coding is required to actually run the experiment. The advantage of this is, that everything stays very flexible and lots of different experiments can be designed.

In [7]:
# initialize variable to save the results
results = None

for screen in experimental_course:

    # use screen meta data to display information to participant
    if screen["block_changed"]:
        whispy.gui.InfoWindow(f"Please rate the {screen['attribute']} next.")

    # start rating screen
    mushra_like_2d = whispy.methods.DragAndDropMUSHRA(
        screen, stimuli_handler, config_attributes, config_mushra_like_2d)

    # update results
    results = mushra_like_2d.get_results(results)

The results were updated iteratively. They contain the ratings in *long* format and are stored in a pandas DataFrame:

In [8]:
print(results)

   block  section  reference  test  block_changed  section_changed  \
0      0        0          1     2           True             True   
1      0        0          1     3           True             True   
2      0        1          1     3          False             True   
3      0        1          1     2          False             True   
4      1        0          1     3           True            False   
5      1        0          1     2           True            False   

                attribute block_name section_name    rating  
0              difference    Block 1    Section 1  0.334711  
1              difference    Block 1    Section 1  0.085744  
2              difference    Block 1    Section 2  0.443182  
3              difference    Block 1    Section 2  0.145661  
4  tone_color_bright-dark    Block 2    Section 3 -0.462810  
5  tone_color_bright-dark    Block 2    Section 3 -0.764463  
